# Lab Day 1 — Notebook

> Notebook thực thi các phần Part 0–4 bằng các module trong `code/`.
> Pipeline bao gồm xử lý dữ liệu, chẩn đoán mô hình, thí nghiệm có kiểm soát, chọn mô hình bằng validation và đánh giá chính thức.
> Kết quả được lưu trong thư mục nộp bài có hậu tố mã sinh viên.

In [ ]:
# ===== Google Colab and GitHub repository =====
import json, math, subprocess, sys
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt

REPO_URL = "https://github.com/VuQuocHuy89/K4-Track4-Day1-Neuralnetwork.git"
REPO_ROOT = Path("/content/k4_day1_lab_source")
if not REPO_ROOT.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
STUDENT_ID = "2A202602929"
SUBMISSION_DIR = REPO_ROOT / f"submission_{STUDENT_ID}"
if (SUBMISSION_DIR / "code/train.py").is_file():
    NOTEBOOK_CODE_DIR = SUBMISSION_DIR / "code"
    OUT_DIR = SUBMISSION_DIR
else:
    NOTEBOOK_CODE_DIR = REPO_ROOT / "code"
    OUT_DIR = Path("/content") / f"submission_{STUDENT_ID}"
RAW_DATA_PATH = REPO_ROOT / "data/covtype.csv.gz"
for path in (RAW_DATA_PATH, REPO_ROOT / "data/split_metadata.csv"):
    if not path.is_file():
        raise FileNotFoundError(path)
sys.path.insert(0, str(NOTEBOOK_CODE_DIR))

for folder in (OUT_DIR / "figures", OUT_DIR / "results"):
    folder.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Repository:", REPO_ROOT, "| data:", RAW_DATA_PATH, "| output:", OUT_DIR)
print("PyTorch:", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, activation_stats
from train import DEFAULT_CFG, set_seed, evaluate, compute_loss, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, to_row, write_xlsx

## Part 0 — Dữ liệu

Pipeline nạp train/eval đã chia theo metadata, tách validation có phân tầng từ train, tính thống kê chuẩn hoá trên train và chuyển tensor lên thiết bị được chọn.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/split_data.py",
        "--data",
        str(RAW_DATA_PATH),
        "--meta",
        str(REPO_ROOT / "data/split_metadata.csv"),
        "--out",
        str(REPO_ROOT / "data/processed"),
    ],
    cwd=REPO_ROOT,
    check=True,
)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=str(REPO_ROOT / "data/processed"))
X_tr, y_tr = data["X_tr"], data["y_tr"]
X_val, y_val = data["X_val"], data["y_val"]
print("Shapes:", tuple(X_tr.shape), tuple(X_val.shape), tuple(data["X_eval"].shape))
train_mean = X_tr[:, :10].mean(dim=0)
train_std = X_tr[:, :10].std(dim=0, correction=0)
print("Train numeric mean max abs:", float(train_mean.abs().max()), "| std range:", float(train_std.min()), float(train_std.max()))
assert torch.allclose(train_mean, torch.zeros_like(train_mean), atol=1e-4)
assert torch.allclose(train_std, torch.ones_like(train_std), atol=1e-3)

## Part 1 — Mô hình và chẩn đoán ban đầu

Kiến trúc M-base theo yêu cầu là `54 → 256 → 128 → 7`, gồm 47.879 tham số. Mô hình trả về logits thô có shape `(B, 7)`.

In [ ]:
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)]

shape_probe = torch.randn(8, 54, device=device)
for layer_index, layer in enumerate(model.net):
    shape_probe = layer(shape_probe)
    print(f"layer {layer_index + 1} ({type(layer).__name__}): {tuple(shape_probe.shape)}")
assert shape_probe.shape == (8, 7)
print("M-base params:", count_params(model), "| output shape:", tuple(shape_probe.shape))

step0 = evaluate(model, X_val, y_val, loss_name="ce")
print(f"Step-0 validation CE: {step0['loss']:.6f}; ln(7)={math.log(7):.6f}")

for init_name in ("he", "zeros"):
    set_seed(1)
    diagnostic_model = MLP(hidden=(256, 128), dropout=0.0, init=init_name).to(device)
    stds = activation_stats(diagnostic_model, X_val[:1024])
    init_loss = evaluate(diagnostic_model, X_val, y_val, loss_name="ce")["loss"]
    print(f"Initialization {init_name}: hidden-ReLU/logit std={stds}; val CE={init_loss:.6f}")

# Gradient-flow probe on a sufficiently large batch to activate every hidden unit.
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
model.train()
model.zero_grad(set_to_none=True)
probe_loss = compute_loss(model(X_tr[:1024]), y_tr[:1024], "ce")
probe_loss.backward()
gradient_norms = {}
for name, parameter in model.named_parameters():
    norm = None if parameter.grad is None else float(parameter.grad.detach().norm().item())
    gradient_norms[name] = norm
    print(name, "grad_norm=", norm)
assert all(value is not None and value > 0 for value in gradient_norms.values())

# Memorize 20 fixed training examples with regularization disabled.
set_seed(1)
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=1e-2)
x20, y20 = X_tr[:20], y_tr[:20]
overfit_steps, overfit_losses = [], []
for step in range(1, 1001):
    overfit_model.train()
    overfit_optimizer.zero_grad(set_to_none=True)
    overfit_loss = compute_loss(overfit_model(x20), y20, "ce")
    overfit_loss.backward()
    overfit_optimizer.step()
    if step % 10 == 0 or step == 1:
        overfit_model.eval()
        with torch.no_grad():
            check_logits = overfit_model(x20)
            check_loss = float(compute_loss(check_logits, y20, "ce").item())
            check_acc = float((check_logits.argmax(1) == y20).float().mean().item())
        overfit_steps.append(step)
        overfit_losses.append(check_loss)
        if check_loss < 0.01 and check_acc == 1.0:
            break
print(f"20-sample overfit: steps={step}, loss={check_loss:.6f}, accuracy={check_acc:.1%}")
assert check_loss < 0.01 and check_acc == 1.0
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(overfit_steps, overfit_losses)
ax.set(xlabel="Update step", ylabel="CE loss", title="20-sample overfit check")
ax.grid(alpha=0.25)
plt.show()
plt.close(fig)

### Chẩn đoán khởi tạo

Output phía trên ghi nhận loss bước 0, thống kê kích hoạt và kết quả kiểm tra overfit trên 20 mẫu.

## Part 2 — Pipeline huấn luyện và baseline

`run_experiment(cfg, data)` trả về lịch sử theo epoch và phần tổng hợp kết quả. Baseline dùng SGD momentum 0.9, cross-entropy, He initialization, batch size 512 và 20 epoch.

In [ ]:
# Screen each optimizer over the same number of epochs, split, and seed.
optimizer_lr_grid = {
    "sgd": (0.03, 0.1, 0.3),
    "sgd_momentum": (0.03, 0.1, 0.3),
    "adam": (3e-4, 1e-3, 3e-3),
    "adamw": (3e-4, 1e-3, 3e-3),
}
optimizer_predictions = {
    "sgd": "SGD needs an lr that balances stable steps with sufficient progress.",
    "sgd_momentum": "Momentum should improve progress over plain SGD at a suitable lr.",
    "adam": "Adam may reach a stronger validation score in fewer epochs, depending on lr.",
    "adamw": "With weight decay fixed at zero, AdamW should match Adam closely.",
}
screen_results = []
best_optimizer_lr = {}
for optimizer_name, lr_values in optimizer_lr_grid.items():
    print("Prediction before screening:", optimizer_name, "->", optimizer_predictions[optimizer_name])
    optimizer_runs = []
    for lr_value in lr_values:
        lr_tag = f"{lr_value:g}".replace(".", "p")
        exp_id = f"screen-{optimizer_name}-lr{lr_tag}"
        screen_cfg = {
            **DEFAULT_CFG,
            "exp_id": exp_id,
            "group": "optimizer",
            "description": f"5-epoch {optimizer_name} learning-rate screen at lr={lr_value:g}",
            "optimizer": optimizer_name,
            "lr": lr_value,
            "weight_decay": 0.0,
            "epochs": 5,
            "seed": 1,
        }
        result = run_experiment(screen_cfg, data)
        save_result(result, str(OUT_DIR / "results"))
        plot_run(result, str(OUT_DIR / "figures" / f"{exp_id}.png"))
        optimizer_runs.append(result)
        screen_results.append(result)
    valid_optimizer_runs = [
        r for r in optimizer_runs
        if not r["summary"]["diverged"]
        and r["summary"]["val_macro_f1"] is not None
    ]
    if not valid_optimizer_runs:
        raise RuntimeError(f"All {optimizer_name} learning-rate screens diverged.")
    best_screen = max(
        valid_optimizer_runs,
        key=lambda r: r["summary"]["val_macro_f1"],
    )
    best_optimizer_lr[optimizer_name] = float(best_screen["cfg"]["lr"])
    for screened_run in optimizer_runs:
        print(
            f"Observed {screened_run['cfg']['exp_id']}: "
            f"val macro-F1={screened_run['summary']['val_macro_f1']}"
        )
    print(
        f"Selected {optimizer_name} lr={best_optimizer_lr[optimizer_name]:g}; "
        f"5-epoch val macro-F1={best_screen['summary']['val_macro_f1']:.4f}"
    )
for optimizer_name in optimizer_lr_grid:
    runs = [r for r in screen_results if r["cfg"]["optimizer"] == optimizer_name]
    plot_compare(
        runs,
        "val_macro_f1",
        str(OUT_DIR / "figures" / f"compare_lr_{optimizer_name}.png"),
        title=f"Learning-rate screening: {optimizer_name}",
    )
plot_compare(
    screen_results,
    "val_macro_f1",
    str(OUT_DIR / "figures" / "compare_optimizer_lr_screen.png"),
    title="Optimizer learning-rate screening (5 epochs)",
)

chosen_lr = best_optimizer_lr["sgd_momentum"]
cfg = {
    **DEFAULT_CFG,
    "lr": chosen_lr,
    "description": f"Baseline M-base; lr selected on validation={chosen_lr:g}",
}
print(f"Baseline SGD+momentum lr selected by validation: {chosen_lr:g}")

base_results = []
for seed in (1, 2, 3):
    base_cfg = {
        **cfg,
        "exp_id": f"base-s{seed}",
        "group": "baseline",
        "seed": seed,
    }
    result = run_experiment(base_cfg, data)
    save_result(result, str(OUT_DIR / "results"))
    plot_run(result, str(OUT_DIR / "figures" / f"{base_cfg['exp_id']}.png"))
    base_results.append(result)

valid_baselines = [
    r for r in base_results
    if not r["summary"]["diverged"] and r["summary"]["val_macro_f1"] is not None
]
if not valid_baselines:
    raise RuntimeError("No valid baseline run completed.")
baseline_result = next(
    (r for r in valid_baselines if r["cfg"]["seed"] == 1),
    valid_baselines[0],
)
baseline_f1_values = np.asarray(
    [r["summary"]["val_macro_f1"] for r in valid_baselines], dtype=float
)
baseline_acc_values = np.asarray(
    [r["summary"]["val_acc"] for r in valid_baselines], dtype=float
)
baseline_f1_mean = float(baseline_f1_values.mean())
baseline_f1_std = float(baseline_f1_values.std(ddof=1)) if len(valid_baselines) > 1 else None
baseline_acc_mean = float(baseline_acc_values.mean())
baseline_acc_std = float(baseline_acc_values.std(ddof=1)) if len(valid_baselines) > 1 else None
noise_2sigma = None if baseline_f1_std is None else 2.0 * baseline_f1_std
noise_label = f"{noise_2sigma:.4f}" if noise_2sigma is not None else "not measured"
all_results = screen_results + base_results
experiment_notes = {
    r["cfg"]["exp_id"]: r["cfg"].get("description", "") for r in all_results
}
for screened_run in screen_results:
    exp_id = screened_run["cfg"]["exp_id"]
    optimizer_name = screened_run["cfg"]["optimizer"]
    score = screened_run["summary"]["val_macro_f1"]
    observed = (
        f"observed 5-epoch val macro-F1={score:.4f}; used for learning-rate screening only"
        if score is not None
        else "diverged before a validation score was available"
    )
    experiment_notes[exp_id] = (
        f"Prediction: {optimizer_predictions[optimizer_name]} {observed}."
    )

# Compare each optimizer at its validation-selected learning rate using the same seed.
optimizer_full_results = []
for optimizer_name in ("sgd", "adam", "adamw"):
    description = (
        f"{optimizer_name} at lr={best_optimizer_lr[optimizer_name]:g}; "
        "learning rate selected by the common 5-epoch validation screen"
    )
    print("Prediction before full run:", optimizer_name, "->", optimizer_predictions[optimizer_name])
    exp_id = f"opt-{optimizer_name}-bestlr"
    run_cfg = {
        **cfg,
        "exp_id": exp_id,
        "group": "optimizer",
        "description": description,
        "optimizer": optimizer_name,
        "lr": best_optimizer_lr[optimizer_name],
        "weight_decay": 0.0,
        "seed": 1,
        "epochs": 20,
    }
    result = run_experiment(run_cfg, data)
    save_result(result, str(OUT_DIR / "results"))
    plot_run(result, str(OUT_DIR / "figures" / f"{exp_id}.png"))
    optimizer_full_results.append(result)
    score = result["summary"]["val_macro_f1"]
    delta = score - baseline_f1_mean
    evidence = (
        ("vượt" if abs(delta) > noise_2sigma else "trong")
        if noise_2sigma is not None
        else "chưa đo được nhiễu"
    )
    experiment_notes[exp_id] = (
        f"{description}; val macro-F1={score:.4f}; delta vs baseline mean={delta:+.4f}; "
        f"{evidence} 2σ={noise_label}"
        if noise_2sigma is not None
        else f"{description}; val macro-F1={score:.4f}; baseline seed noise unavailable"
    )
    print(
        f"Observed val macro-F1={score:.4f}; delta vs baseline mean={delta:+.4f}; "
        f"comparison {evidence}; 2σ={noise_label}."
    )

all_results.extend(optimizer_full_results)
print(
    f"Baseline val macro-F1: {baseline_f1_mean:.4f} ± {baseline_f1_std:.4f}; "
    f"2σ={noise_2sigma:.4f}"
    if noise_2sigma is not None
    else f"Baseline val macro-F1: {baseline_f1_mean:.4f}; seed noise unavailable."
)
print(
    f"Baseline val accuracy: {baseline_acc_mean:.4f} ± {baseline_acc_std:.4f}"
    if baseline_acc_std is not None
    else f"Baseline val accuracy: {baseline_acc_mean:.4f}; seed noise unavailable."
)

### Kết quả baseline

Ba seed sử dụng cùng split dữ liệu và cấu hình huấn luyện. Output tổng hợp validation metrics qua các seed; biểu đồ thể hiện loss, điểm validation và gradient norm theo epoch.

## Part 3 — Thí nghiệm có kiểm soát

Mỗi cấu hình ghi nhận giả thuyết trước khi huấn luyện và thay đổi một yếu tố so với cấu hình đối chứng. Các chủ đề gồm loss, optimizer, hyperparameter, dropout, clipping, AMP và khởi tạo.

### Kế hoạch và kết quả thí nghiệm

Các cấu hình, giả thuyết và kết quả validation được ghi trong output. Workbook và biểu đồ tổng hợp kết quả theo nhóm thí nghiệm.

In [ ]:
# Predictions are recorded before each intervention; each run changes one factor from its control.
baseline_grad_reference = float(
    np.nanmedian(np.asarray(baseline_result["history"]["grad_norm"], dtype=float))
)
clip_c = max(1e-3, 0.5 * baseline_grad_reference)
high_lr = max(chosen_lr * 10.0, chosen_lr + 1e-3)
EXPERIMENT_PLAN = [
    {
        "exp_id": "loss-mse",
        "group": "loss",
        "overrides": {"loss": "mse"},
        "prediction": "MSE trên logits có thể hội tụ chậm hơn cross-entropy; so sánh macro-F1, không so trực tiếp loss.",
    },
    {
        "exp_id": "hparam-batch-1024",
        "group": "hparam",
        "overrides": {"batch": 1024},
        "prediction": "Batch lớn hơn giảm số bước cập nhật mỗi epoch; thời gian và macro-F1 có thể thay đổi.",
    },
    {
        "exp_id": "sgdm-cosine",
        "group": "hparam",
        "overrides": {"scheduler": "cosine"},
        "prediction": "Cosine decay có thể giảm dao động cuối quá trình khi giữ nguyên optimizer, lr ban đầu và số epoch.",
    },
    {
        "exp_id": "sgdm-40-epochs",
        "group": "hparam",
        "overrides": {},
        "epochs": 40,
        "prediction": "Thêm epoch có thể cải thiện macro-F1 nếu validation còn tiến triển; cũng có thể tăng overfitting.",
    },
    {
        "exp_id": "adamw-wd-001",
        "group": "hparam",
        "overrides": {
            "optimizer": "adamw",
            "lr": best_optimizer_lr["adamw"],
            "weight_decay": 0.01,
        },
        "prediction": "Weight decay 0,01 có thể hạn chế trọng số lớn; so với AdamW cùng lr và seed nhưng weight decay bằng 0.",
    },
    {
        "exp_id": "dropout-03",
        "group": "dropout",
        "overrides": {"dropout": 0.3},
        "prediction": "Nếu baseline chưa quá khớp, dropout có thể không cải thiện macro-F1 và làm train loss cao hơn.",
    },
    {
        "exp_id": "clip-highlr-none",
        "group": "clipping",
        "overrides": {"lr": high_lr, "clip_norm": None},
        "prediction": "Learning rate cao có thể tạo cập nhật quá lớn và làm loss dao động.",
    },
    {
        "exp_id": "clip-highlr-c",
        "group": "clipping",
        "overrides": {"lr": high_lr, "clip_norm": clip_c},
        "prediction": "Cùng learning rate cao, clipping có thể hạn chế cập nhật quá lớn khi ngưỡng được kích hoạt.",
    },
    {
        "exp_id": "amp-fp16",
        "group": "amp",
        "overrides": {"precision": "fp16"},
        "prediction": "FP16 có thể giảm bộ nhớ GPU; tốc độ có thể không tăng trên MLP nhỏ.",
    },
    {
        "exp_id": "amp-bf16",
        "group": "amp",
        "overrides": {"precision": "bf16"},
        "prediction": "BF16 có dải biểu diễn rộng hơn FP16 và thường không cần gradient scaler.",
    },
    {
        "exp_id": "init-zeros",
        "group": "init",
        "overrides": {"init": "zeros"},
        "prediction": "Khởi tạo zeros giữ neuron ẩn đối xứng và ReLU(0)=0, nên các lớp ẩn không học đặc trưng riêng.",
    },
]

experiment_results = []
skipped_experiments = {}
for plan in EXPERIMENT_PLAN:
    exp_id = plan["exp_id"]
    if exp_id == "amp-fp16" and device.type != "cuda":
        skipped_experiments[exp_id] = "FP16 cần CUDA; thiết bị hiện tại không có CUDA."
        print("Skipping", exp_id, "-", skipped_experiments[exp_id])
        continue
    if (
        exp_id == "amp-bf16"
        and device.type == "cuda"
        and not torch.cuda.is_bf16_supported()
    ):
        skipped_experiments[exp_id] = "GPU hiện tại không hỗ trợ BF16."
        print("Skipping", exp_id, "-", skipped_experiments[exp_id])
        continue

    print("Prediction before run:", exp_id, "->", plan["prediction"])
    epochs = int(plan.get("epochs", 20))
    exp_cfg = {
        **cfg,
        **plan["overrides"],
        "exp_id": exp_id,
        "group": plan["group"],
        "seed": 1,
        "epochs": epochs,
        "description": plan["prediction"],
    }
    result = run_experiment(exp_cfg, data)
    save_result(result, str(OUT_DIR / "results"))
    plot_run(result, str(OUT_DIR / "figures" / f"{exp_id}.png"))
    experiment_results.append(result)
    score = result["summary"]["val_macro_f1"]
    if score is None:
        experiment_notes[exp_id] = (
            f"Prediction: {plan['prediction']} Observed: diverged before a "
            "validation score was available."
        )
        print("Observed: run diverged before a validation score was available.")
    else:
        delta = float(score) - baseline_f1_mean
        evidence = (
            ("vượt" if abs(delta) > noise_2sigma else "trong")
            if noise_2sigma is not None
            else "chưa đo được nhiễu"
        )
        extra = ""
        if plan["group"] == "clipping" and result["summary"].get("clip_fraction") is not None:
            extra = f" Clipping activated on {result['summary']['clip_fraction']:.1%} of finite-gradient steps."
        experiment_notes[exp_id] = (
            f"Prediction: {plan['prediction']} Observed val macro-F1={score:.4f}; "
            f"delta vs baseline mean={delta:+.4f}; {evidence}; 2σ={noise_label}."
            f"{extra}"
        )
        print(
            f"Observed val macro-F1={score:.4f}; delta vs baseline mean={delta:+.4f}; "
            f"{evidence}; 2σ={noise_label}."
        )
        if extra:
            print(extra.strip())
    print(
        "Best epoch:", result["summary"]["best_epoch"],
        "| diverged:", result["summary"]["diverged"],
    )
all_results.extend(experiment_results)

### So sánh trên validation

Biểu đồ và workbook tổng hợp validation metrics, mức thay đổi so với baseline và độ nhiễu đo được giữa các seed.

In [ ]:
results_by_id = {result["cfg"]["exp_id"]: result for result in all_results}
comparison_sets = {
    "loss": [baseline_result, results_by_id.get("loss-mse")],
    "optimizer": [baseline_result] + optimizer_full_results,
    "hparam": [baseline_result] + [
        result for result in experiment_results if result["cfg"]["group"] == "hparam"
    ],
    "weight_decay": [
        results_by_id.get("opt-adamw-bestlr"),
        results_by_id.get("adamw-wd-001"),
    ],
    "dropout": [baseline_result, results_by_id.get("dropout-03")],
    "clipping": [results_by_id.get("clip-highlr-none"), results_by_id.get("clip-highlr-c")],
    "amp": [baseline_result] + [
        result for result in experiment_results if result["cfg"]["group"] == "amp"
    ],
    "init": [baseline_result, results_by_id.get("init-zeros")],
}
for group, group_results in comparison_sets.items():
    group_results = [result for result in group_results if result is not None]
    if len(group_results) >= 2:
        plot_compare(
            group_results,
            "val_macro_f1",
            str(OUT_DIR / "figures" / f"compare_{group}.png"),
            title=f"Validation macro-F1: {group}",
        )
print("Saved validation comparison plots.")

## Part 4 — Đánh giá cuối trên eval và tổng hợp kết quả

Cấu hình cuối được chọn bằng macro-F1 trên validation. Evaluator chính thức tính accuracy, macro-F1, chỉ số theo lớp và ma trận nhầm lẫn cho baseline cùng cấu hình cuối.

In [ ]:
# Select the final run from full-length validation results only.
# Eval labels are used only after this selection.
final_candidates = [
    result
    for result in all_results
    if result["cfg"]["epochs"] >= 20
    and not result["summary"]["diverged"]
    and result["best_state"] is not None
    and result["summary"]["val_macro_f1"] is not None
]
if not final_candidates:
    raise RuntimeError("No valid full-length run is available for validation-based selection.")
result_final = max(
    final_candidates,
    key=lambda result: result["summary"]["val_macro_f1"],
)
cfg_final = result_final["cfg"]
print(
    "Final config selected by validation:",
    cfg_final["exp_id"],
    "| optimizer:", cfg_final["optimizer"],
    "| lr:", cfg_final["lr"],
    "| epochs:", cfg_final["epochs"],
    "| val macro-F1:", result_final["summary"]["val_macro_f1"],
    "| best epoch:", result_final["summary"]["best_epoch"],
)

# Official evaluation is run for the baseline and selected final configuration only.
eval_scores_by_id = {}
def _run_official_eval(result, role):
    exp_id = result["cfg"]["exp_id"]
    if exp_id in eval_scores_by_id:
        return eval_scores_by_id[exp_id]
    if role == "final":
        pred_path = OUT_DIR / "predictions_eval.csv"
        json_path = OUT_DIR / "eval_result.json"
    else:
        pred_path = OUT_DIR / "results" / "baseline_predictions_eval.csv"
        json_path = OUT_DIR / "results" / "baseline_eval_result.json"
    final_eval(result["cfg"], result, data, str(pred_path))
    completed = subprocess.run(
        [
            sys.executable,
            "scripts/evaluate.py",
            "--pred",
            str(pred_path),
            "--out",
            str(json_path),
        ],
        cwd=REPO_ROOT,
        check=True,
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    with json_path.open(encoding="utf-8") as stream:
        scores = json.load(stream)
    eval_scores_by_id[exp_id] = scores
    return scores

if baseline_result["cfg"]["exp_id"] == result_final["cfg"]["exp_id"]:
    final_eval_scores = _run_official_eval(result_final, "final")
    baseline_eval_scores = final_eval_scores
else:
    baseline_eval_scores = _run_official_eval(baseline_result, "baseline")
    final_eval_scores = _run_official_eval(result_final, "final")
print("Baseline official eval:", baseline_eval_scores["accuracy"], baseline_eval_scores["macro_f1"])
print("Final official eval:", final_eval_scores["accuracy"], final_eval_scores["macro_f1"])

In [ ]:
print("Final model per-class scores (from official eval_result.json):")
for row in final_eval_scores['per_class']:
    print(f"class {row['cls']}: support={row['support']}, precision={row['precision']:.4f}, recall={row['recall']:.4f}, F1={row['f1']:.4f}")
confusion = np.asarray(final_eval_scores['confusion_matrix'], dtype=np.int64)
print("Confusion matrix (rows=true class, columns=predicted class):\n", confusion)
worst_class = min(final_eval_scores['per_class'], key=lambda row: row['f1'])['cls']
confused_counts = confusion[worst_class].copy(); confused_counts[worst_class] = 0
most_confused_class = int(confused_counts.argmax())
print(f"Lowest-F1 class: {worst_class}; most often predicted as class {most_confused_class} ({int(confused_counts[most_confused_class])} samples). ")

### Chỉ số theo lớp và ma trận nhầm lẫn

Output liệt kê support, precision, recall, F1 của từng lớp và số lượng dự đoán nhầm theo cặp nhãn thật–nhãn dự đoán.

In [ ]:
table_rows = []
for result in all_results:
    exp_id = result['cfg']['exp_id']
    table_rows.append(to_row(result, eval_scores=eval_scores_by_id.get(exp_id), notes=experiment_notes.get(exp_id, result['cfg'].get('description', ''))))
xlsx_path = OUT_DIR / 'experiments.xlsx'
template_path = REPO_ROOT / 'templates/experiment_table_template.xlsx'
write_xlsx(table_rows, str(template_path), str(xlsx_path))

# The workbook template contains formulas in the Seeds and Summary sheets.
from openpyxl import load_workbook
workbook = load_workbook(xlsx_path)
seed_sheet = workbook['Seeds']
valid_seed_ids = [r['cfg']['exp_id'] for r in valid_baselines]
for row_index in range(2, seed_sheet.max_row + 1):
    seed_sheet.cell(row_index, 1).value = valid_seed_ids[row_index - 2] if row_index - 2 < len(valid_seed_ids) else None
summary_sheet = workbook['Summary']
for row_index in range(2, summary_sheet.max_row + 1):
    group = summary_sheet.cell(row_index, 1).value
    if group is None: continue
    if group == 'baseline':
        note = f"{len(valid_baselines)} valid seeds; val macro-F1 mean={baseline_f1_mean:.4f}; 2σ={noise_2sigma:.4f}" if noise_2sigma is not None else f"{len(valid_baselines)} valid seed; seed noise not measured"
    else:
        group_runs = [r for r in all_results if r['cfg'].get('group') == group and r['cfg'].get('epochs', 0) >= 20 and r['summary'].get('val_macro_f1') is not None]
        if group_runs:
            best_group = max(group_runs, key=lambda r: r['summary']['val_macro_f1'])
            delta = best_group['summary']['val_macro_f1'] - baseline_f1_mean
            noise_note = 'seed noise unavailable' if noise_2sigma is None else ('difference exceeds 2σ' if abs(delta) > noise_2sigma else 'difference is within 2σ')
            note = f"Best full run: {best_group['cfg']['exp_id']} val macro-F1={best_group['summary']['val_macro_f1']:.4f}; delta vs baseline mean={delta:+.4f}; {noise_note}."
        elif group == 'amp' and 'amp-fp16' in skipped_experiments:
            note = skipped_experiments['amp-fp16']
        else:
            note = 'No completed run for this topic.'
    summary_sheet.cell(row_index, 8).value = note
workbook.save(xlsx_path)
print("Experiment outputs:", OUT_DIR)